In [1]:
pip install pandas pyarrow


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install endee


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip show endee

Name: endee
Version: 0.1.13
Summary: Endee is the Next-Generation Vector Database for Scalable, High-Performance AI
Home-page: https://endee.io
Author: Endee Labs
Author-email: dev@endee.io
License: 
Location: /home/debian/latest_VDB/VectorDBBench/venv2/lib/python3.11/site-packages
Requires: httpx, msgpack, numpy, orjson, pydantic, requests
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
base_path = "/home/debian/latest_VDB/VectorDBBench/vectordataset_lte"
dataset_folder = "cohere/cohere_medium_1m"

In [3]:
#For checking parquet file contents
import pyarrow.parquet as pq
import os

file_name = "shuffle_train.parquet"  #input

# Build full path
file_path = os.path.join(base_path, dataset_folder, file_name)


parquet_file = pq.ParquetFile(file_path)

# Read only first batch of rows
first_batch = next(parquet_file.iter_batches(batch_size=5))
preview = first_batch.to_pandas()

for col in preview.columns:
    if preview[col].dtype == object and isinstance(preview[col].iloc[0], list):
        preview[col] = preview[col].apply(lambda x: x[:5] if x is not None else x)

print(preview)

       id                                                emb
0  322406  [0.19600096, -0.5270862, -0.29519123, 0.429556...
1  337610  [0.32829463, -0.055560444, -0.06252914, -0.101...
2  714728  [-0.054155815, 0.009554057, 0.24696879, -0.142...
3  354737  [0.2501778, 0.017853737, -0.43795395, 0.522256...
4  290979  [0.3444257, -0.62243223, -0.20940691, -0.08620...


In [4]:
from endee import Endee
client = Endee(token="localtest")
client.set_base_url("http://148.113.54.173:8080/api/v1")

In [5]:
#For checking indexes
for obj in client.list_indexes().get('indexes'):
    print(obj['name'])
    print(obj['total_elements'])
    print('\t')

test_1M_vaib_labelfilter_0503_1
1000000
	
test_1M_vaib_labelfilter_0503_1_adup1
1000000
	
test_1M_vaib_labelfilter_0503_1_adup3
1000000
	
test_1M_vaib_labelfilter_0503_1_adup4
1000000
	
test_1M_vaib_labelfilter_0503_1_adup5
200000
	
test_1M_vaib_labelfilter_0503_1_adup6
10000
	


In [6]:
#give the index name
index_name = "test_1M_vaib_labelfilter_0503_1_adup6"
index = client.get_index(index_name)

In [7]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_0503_1_adup6',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 10000,
 'precision': 'int16',
 'M': 16}

In [8]:
#For querying (int filter)- lte filter
import pyarrow.parquet as pq
import os

#inputs
int_rate_query = "1p"   # change to 1p, 50p, 80p, 99p
top_k = 30

# Mode 1: Single query id
# Mode 2: Find all query ids with recall < 1
mode = 2
query_id = 457    # only used in mode 1

# Map int_rate_query to lte threshold
lte_map = {
    "1p":  9999,
    "50p": 499999,
    "80p": 799999,
    "99p": 989999,
}
lte_threshold = lte_map[int_rate_query]

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
lte_dataset_path = os.path.join("/home/debian/latest_VDB/VectorDBBench/vectordataset_lte", dataset_folder)
test_parquet_path = os.path.join(dataset_path, "test.parquet")
gt_parquet_path = os.path.join(lte_dataset_path, f"neighbors_int_{int_rate_query}.parquet")

# Read parquet files
test_df = pq.ParquetFile(test_parquet_path).read().to_pandas()
gt_df = pq.ParquetFile(gt_parquet_path).read().to_pandas()

def run_query(query_id):
    query_row = test_df[test_df["id"] == query_id]
    if query_row.empty:
        print(f"Query ID {query_id} not found in test.parquet")
        return None

    query_vector = query_row["emb"].values[0]

    gt_row = gt_df[gt_df["id"] == query_id]
    if gt_row.empty:
        print(f"Query ID {query_id} not found in ground truth file")
        return None

    ground_truth = gt_row["neighbors_id"].values[0][:top_k]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        filter=[{"id": {"$lte": lte_threshold}}],
        include_vectors=False,
    )
    returned_ids = [int(r["id"]) for r in results]

    intersection = len(set(returned_ids) & set(ground_truth))
    recall = intersection / len(ground_truth)

    return returned_ids, ground_truth, recall


if mode == 1:
    result = run_query(query_id)
    if result:
        returned_ids, ground_truth, recall = result
        print(f"Query ID: {query_id}")
        print(f"Returned IDs: {','.join(map(str, returned_ids))}")
        print(f"Ground Truth: {','.join(map(str, ground_truth))}")
        print(f"Recall: {recall:.4f}")

elif mode == 2:
    print(f"Finding all query IDs with recall < 1.0 ...\n")
    low_recall_ids = []
    all_recalls = []

    for qid in test_df["id"].values:
        result = run_query(qid)
        if result:
            returned_ids, ground_truth, recall = result
            all_recalls.append(recall)
            if recall < 1.0:
                low_recall_ids.append(qid)
                print(f"Query ID: {qid}")
                print(f"Returned IDs: {','.join(map(str, returned_ids))}")
                print(f"Ground Truth: {','.join(map(str, ground_truth))}")
                print(f"Recall: {recall:.4f}\n")

    print(f"Total query IDs with recall < 1.0: {len(low_recall_ids)}")
    print(f"IDs: {low_recall_ids}")
    print(f"Average Recall across all {len(all_recalls)} queries: {sum(all_recalls)/len(all_recalls):.4f}")

Finding all query IDs with recall < 1.0 ...

Query ID: 0
Returned IDs: 
Ground Truth: 940,707,7629,476,9354,8339,9711,7862,3526,3651,5672,1569,4213,8423,7248,7091,536,6181,5945,7663,7193,5093,3532,5840,6714,9207,4581,5040,4688,9511
Recall: 0.0000

Query ID: 1
Returned IDs: 
Ground Truth: 7793,2368,525,646,4998,181,9359,7788,987,7515,5872,4948,6100,9093,3315,6643,8754,7594,40,1885,4400,7870,312,179,6219,4729,4132,8620,4586,285
Recall: 0.0000

Query ID: 2
Returned IDs: 2927,91,6170
Ground Truth: 4031,6525,1838,608,1490,2174,5885,8649,4497,9138,8227,4707,2927,2838,290,3707,3474,3738,2848,6854,3622,3013,5110,3755,4974,1683,7348,2012,6511,934
Recall: 0.0333

Query ID: 3
Returned IDs: 9796
Ground Truth: 3745,3136,208,8661,9742,9191,7532,129,3890,9038,3477,2241,8561,9796,9246,1416,7621,4083,6538,7052,2527,7476,5862,7923,112,5852,3831,2187,8995,7554
Recall: 0.0333

Query ID: 4
Returned IDs: 
Ground Truth: 7968,2957,9161,8580,5436,7899,5870,4256,2141,4904,2942,85,2078,9288,6915,1963,5662,1548,7

In [8]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_0503_1_adup6',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 10000,
 'precision': 'int16',
 'M': 16}

In [12]:
#For deleting (int filter) - lte
int_rate_delete = "99p"  # change to 1p, 20p, 50p, 99p

lte_map = {
    "1p":  990000,
    "20p": 800000,
    "50p": 500000,
    "99p": 10000,
}
lte_threshold = lte_map[int_rate_delete]

result = index.delete_with_filter([{"id": {"$gte": lte_threshold}}])
print(f"Deleted vectors with id >= {lte_threshold}")
print(result)

Deleted vectors with id >= 10000
990000 vectors deleted


In [11]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_0503_1_adup6',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 10000,
 'precision': 'int16',
 'M': 16}

In [12]:
#For reinserting deleted vectors (int filter - lte)
import pyarrow.parquet as pq
import os
import time

# User inputs
int_rate_insert = "99p"  # change to 1p, 20p, 50p, 99p
batch_size = 1000

# Map int_rate to gte threshold (same as delete)
lte_map = {
    "1p":  990000,
    "20p": 800000,
    "50p": 500000,
    "99p": 10000,
}
lte_threshold = lte_map[int_rate_insert]

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
emb_path = os.path.join(dataset_path, "shuffle_train.parquet")

# Process shuffle_train in batches to avoid RAM issues
emb_file = pq.ParquetFile(emb_path)
total_inserted = 0

start = time.perf_counter()
for batch in emb_file.iter_batches(batch_size=batch_size):
    batch_df = batch.to_pandas()

    # Filter by id >= lte_threshold
    batch_df = batch_df[batch_df["id"] >= lte_threshold]

    if batch_df.empty:
        continue

    batch_vectors = []
    for _, row in batch_df.iterrows():
        vector_data = {
            "id": str(row["id"]),
            "vector": row["emb"],
            "meta": {"id": row["id"]},
            "filter": {
                "id": row["id"],
            }
        }
        batch_vectors.append(vector_data)

    index.upsert(batch_vectors)
    total_inserted += len(batch_vectors)
    print(f"Upserted {len(batch_vectors)} vectors, total so far: {total_inserted}")

end = time.perf_counter()
print(f"Done! Total inserted: {total_inserted} vectors with id >= {lte_threshold}")
print("Time taken:", end - start)

Upserted 991 vectors, total so far: 991
Upserted 993 vectors, total so far: 1984
Upserted 989 vectors, total so far: 2973
Upserted 991 vectors, total so far: 3964
Upserted 992 vectors, total so far: 4956
Upserted 991 vectors, total so far: 5947
Upserted 990 vectors, total so far: 6937
Upserted 986 vectors, total so far: 7923
Upserted 996 vectors, total so far: 8919
Upserted 991 vectors, total so far: 9910
Upserted 993 vectors, total so far: 10903
Upserted 992 vectors, total so far: 11895
Upserted 989 vectors, total so far: 12884
Upserted 995 vectors, total so far: 13879
Upserted 988 vectors, total so far: 14867
Upserted 992 vectors, total so far: 15859
Upserted 988 vectors, total so far: 16847
Upserted 988 vectors, total so far: 17835
Upserted 996 vectors, total so far: 18831
Upserted 993 vectors, total so far: 19824
Upserted 993 vectors, total so far: 20817
Upserted 989 vectors, total so far: 21806
Upserted 993 vectors, total so far: 22799
Upserted 982 vectors, total so far: 23781
Ups

In [10]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_0503_1_adup6',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 10000,
 'precision': 'int16',
 'M': 16}